# ConformerFlow — Data Preparation (Run Once)

**Purpose:** Download and parse ALL NMR structures from RCSB PDB (~17,000 entries).
Saves parsed data as Kaggle Dataset for use in all future training sessions.

**Run this notebook once on your second Kaggle account.**

**Settings → Accelerator → None** (no GPU needed for data prep)
**Settings → Turn on Internet → ON**

In [ ]:
# Cell 1 — Clone repo and install
!git clone https://github.com/kumar23kan/conformerflow.git /kaggle/working/conformerflow
!pip install -q biopython gemmi requests tqdm numpy einops scipy

In [ ]:
# Cell 2 — Set paths
import os, sys
os.chdir('/kaggle/working/conformerflow/files')
sys.path.insert(0, '/kaggle/working/conformerflow/files')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 3 — Create directories
from pathlib import Path
for d in ['pdb_data/nmr', 'pdb_data/parsed_nmr', 'pdb_data/splits']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Done')

In [ ]:
# Cell 4 — Download ALL NMR structures (~17,000 entries, ~2 hrs)
from data.fetch_pdb import fetch_nmr_ids, batch_download
ids = fetch_nmr_ids(min_conformers=5, max_results=20000)
print(f'Found {len(ids)} NMR entries')
batch_download(ids, Path('pdb_data/nmr'), max_workers=8)

In [ ]:
# Cell 5 — Parse all structures (~1.5 hrs)
!python3 data/parse_nmr.py \
    --nmr_dir pdb_data/nmr \
    --output_dir pdb_data/parsed_nmr \
    --min_conformers 5 \
    --max_residues 300

In [ ]:
# Cell 6 — Filter and split
!python3 data/filter.py \
    --parsed_dir pdb_data/parsed_nmr \
    --output_dir pdb_data/splits

In [ ]:
# Cell 7 — Check results
import json, os

for split in ['train', 'val', 'test']:
    path = f'pdb_data/splits/{split}.json'
    if os.path.exists(path):
        data = json.load(open(path))
        print(f'{split}: {len(data)} entries')

npz_count = len(list(Path('pdb_data/parsed_nmr').glob('*.npz')))
print(f'Total parsed: {npz_count} .npz files')

In [ ]:
# Cell 8 — Save to Kaggle output
import shutil

# Copy parsed data and splits to output
shutil.copytree(
    'pdb_data/parsed_nmr',
    '/kaggle/working/conformerflow_data/parsed_nmr',
    dirs_exist_ok=True
)
shutil.copytree(
    'pdb_data/splits',
    '/kaggle/working/conformerflow_data/splits',
    dirs_exist_ok=True
)
print('Data saved to output!')
print('parsed_nmr:', len(os.listdir('/kaggle/working/conformerflow_data/parsed_nmr')), 'files')
print('splits:', os.listdir('/kaggle/working/conformerflow_data/splits'))